    Instructions:
    1. Run task_0a.py to generate the vector database if not already generated.
    2. Press Run All (or restart kernel and run all cells).
    3. You will be prompted to provide input values.
    
    Task 6 (LS4): Implement a program which, (a) given one of the feature models and (b) a value k,
    – creates (and saves) an image-image similarity matrix,
    – performs a user selected dimensionality reduction technique (SVD, NNMF, LDA, k-means) on this image-image
    similarity matrix
    – stores the latent semantics in a properly named output file
    – lists image-weight pairs, ordered in decreasing order of weights

In [1]:
FEATURE_SPACE = input("Provide a feature space [color, hog, avgpool, layer3, fc, resnet_output].")

DIM_REDUCTION = input("Provide a dimensionality reduction technique [svd, nnmf, lda, kmeans].")

K = int(input("Enter K, the top K latent semantics to extract for the selected feature space."))


In [2]:
from utils.database_utils import retrieve
feature_vectors = retrieve(f'{FEATURE_SPACE}.pt')

print("Generating top-", K, " image latent semantics under ", FEATURE_SPACE, " feature space using: ", DIM_REDUCTION)

Generating top- 10  image latent semantics under  fc  feature space using:  svd


In [3]:
from utils.database_utils import exists, store, compressed_retrieve, compressed_store
from feature_models.feature_matrix.image_image_similarity import ImageImageSimilarity

if exists(f'img_img_{FEATURE_SPACE}.pt'):

    image_feature_vectors = compressed_retrieve(f'img_img_{FEATURE_SPACE}.pt')

else:
    print('Image-Image similiarity matrix for ', FEATURE_SPACE, ' does not exist, creating one.')

    image_similarity_generator = ImageImageSimilarity(feature_vectors)
    image_feature_vectors = image_similarity_generator.get_matrix()

    compressed_store(image_feature_vectors, f'img_img_{FEATURE_SPACE}.pt')

print("Image-Image similarity matrix for image-id 0 as example:\n")
print(image_feature_vectors[0])
print("Shape: ", image_feature_vectors[0][1].shape)


Image-Image similiarity matrix for  fc  does not exist, creating one.

 Saving:  img_img_fc.pt 

Image-Image similarity matrix for image-id 0 as example:

('Faces', array([1.        , 0.22093663, 0.73819107, ..., 0.30177835, 0.20154907,
       0.15875849]))
Shape:  (4339,)


In [4]:
if DIM_REDUCTION == "svd":
    from feature_reducers.svd import SVDReducer
    reducer = SVDReducer

elif DIM_REDUCTION == "nnmf":
    from feature_reducers.nnmf import NNMFReducer
    reducer = NNMFReducer

elif DIM_REDUCTION == "lda":
    from feature_reducers.lda import LDAReducer
    reducer = LDAReducer

else:
    # kmeans.
    from feature_reducers.kmeans import KMeansReducer
    reducer = KMeansReducer

reducer = reducer(image_feature_vectors, K)

similarity_matrix = reducer.get_similarity_matrix(image_feature_vectors)

latent_semantics = reducer.reduce_features(image_feature_vectors)
print("Top K latent semantics: ")
print(latent_semantics)
print("Shape: ", latent_semantics.shape)

Top K latent semantics: 
[[ 7.13702068  9.0789789   0.03725192 ... -0.69882272 -0.58468877
   0.02573965]
 [ 3.49423577  5.45995749  0.46352612 ... -0.32865251  0.28704614
   0.02264247]
 [ 7.50995963 10.41956002  0.12842626 ... -0.37133821 -0.35190351
   0.03290968]
 ...
 [ 6.35015837  4.32907137  0.28925199 ...  0.03464748 -1.38831915
   0.38968474]
 [ 7.51823265  3.82428052 -0.13733931 ... -1.5847331  -0.15334096
   0.09288571]
 [ 4.52990151  2.88578033 -0.53877476 ... -0.98239053 -0.17443334
   0.21955362]]
Shape:  (4339, 10)


In [5]:
store(reducer, f'LS4_{FEATURE_SPACE}_{DIM_REDUCTION}_reducer.pt')


 Saving:  LS4_fc_svd_reducer.pt 



In [6]:
# List image-weight pairs, ordered in decreasing order of weights

# We are to showcase which image contributes more to each latent feature.
# This is taking the object-feature factor matrix, and sorting by each
# latent feature's weight.

image_id = [feature_tuple[0] for feature_tuple in image_feature_vectors.values()]

image_weight_tuples = list(zip(image_id, similarity_matrix))

print("Image - weight pairs sorted in descending order of weights for each latent feature:")

for i in range(K):
    print("\n\nLatent feature: ", i + 1)
    for IMG_ID, weight in sorted(image_weight_tuples, key=lambda x : x[1][i], reverse=True):
        print("(ID: ", IMG_ID, ", Weight: ", weight[i], end="),\t")

Image - weight pairs sorted in descending order of weights for each latent feature:


Latent feature:  1
(ID:  airplanes , Weight:  0.031536953715164615),	(ID:  airplanes , Weight:  0.03117960949001642),	(ID:  airplanes , Weight:  0.03116135316489022),	(ID:  airplanes , Weight:  0.030833064893807625),	(ID:  airplanes , Weight:  0.0307748558191258),	(ID:  airplanes , Weight:  0.03075024291154925),	(ID:  airplanes , Weight:  0.03067505429308021),	(ID:  airplanes , Weight:  0.030607982964791194),	(ID:  airplanes , Weight:  0.030507254194160227),	(ID:  airplanes , Weight:  0.030482839383515693),	(ID:  airplanes , Weight:  0.030467482161271344),	(ID:  airplanes , Weight:  0.030437798534009967),	(ID:  airplanes , Weight:  0.030381634888654363),	(ID:  airplanes , Weight:  0.03020481148480656),	(ID:  airplanes , Weight:  0.030188235484145242),	(ID:  airplanes , Weight:  0.03018190092629746),	(ID:  airplanes , Weight:  0.03017142109195348),	(ID:  airplanes , Weight:  0.030135689961204772),	(ID: